# YOLO to ONNX Export and Validation

This notebook demonstrates how to export a YOLO model to ONNX format and validate its performance on a test set. The workflow includes:
- Exporting a trained YOLO model to ONNX
- Validating both the original YOLO and exported ONNX models on the same test set
- Comparing the evaluation metrics to ensure consistency and reliability

**Note:** This notebook follows best practices for code structure, naming conventions, and documentation.

In [ ]:
# Install required packages
%pip install onnx onnxruntime ultralytics opencv-python numpy

In [ ]:
from ultralytics import YOLO
import onnx
import onnxruntime as ort
import numpy as np
import cv2
import os
from typing import Tuple, Dict

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
class Config:
    """
    Configuration for model paths and evaluation settings.
    """
    # Path to the trained YOLO model weights
    MODEL_PATH = '../../training/models/YOLO26m_Batch4_March_Dataset/weights/best.pt'
    # Path to the dataset YAML file
    DATA_YAML = '../../configs/dataset.yaml'
    # Path to the test images directory
    TEST_IMAGES_DIR = '../../datasets/batch4/images/test/'
    # Path to the ONNX export output
    ONNX_EXPORT_PATH = 'exported_model.onnx'
    # Inference image size
    IMG_SIZE = 640
    # Confidence and IoU thresholds for evaluation
    CONF_THRESHOLD = 0.20
    IOU_THRESHOLD = 0.45
    # Maximum detections per image
    MAX_DET = 300
    # Use half precision (FP16) for YOLO inference
    HALF = False

config = Config()
print('Configuration initialized.')

In [ ]:
# 1. Load YOLO model and export to ONNX

def export_yolo_to_onnx(model_path: str, export_path: str, img_size: int = 640) -> str:
    """
    Export a YOLO model to ONNX format.
    Args:
        model_path (str): Path to YOLO weights (.pt).
        export_path (str): Output ONNX file path.
        img_size (int): Inference image size.
    Returns:
        str: Path to exported ONNX file.
    """
    model = YOLO(model_path)
    onnx_path = model.export(
        format="onnx",
        imgsz=img_size,
        half=False,
        dynamic=False,
        simplify=True,
        opset=17,
        nms=False,
        batch=1,
        device="cpu",
        path=export_path
    )
    print(f"Exported ONNX model to: {onnx_path}")
    return onnx_path

onnx_path = export_yolo_to_onnx(config.MODEL_PATH, config.ONNX_EXPORT_PATH, config.IMG_SIZE)

In [ ]:
# 2. Validate the exported ONNX model

def validate_onnx_model(onnx_path: str) -> None:
    """
    Validate the exported ONNX model for correctness.
    Args:
        onnx_path (str): Path to the ONNX model file.
    """
    print("Validating ONNX model...")
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print("ONNX model is valid.")

validate_onnx_model(onnx_path)

In [ ]:
# 3. Evaluate YOLO and ONNX models on the test set

def load_test_images(test_dir: str, img_size: int) -> Tuple[np.ndarray, list]:
    """
    Load and preprocess test images from a directory.
    Args:
        test_dir (str): Directory containing test images.
        img_size (int): Target image size (square).
    Returns:
        Tuple[np.ndarray, list]: Batch of images and list of filenames.
    """
    image_files = [f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    images = []
    for fname in image_files:
        img = cv2.imread(os.path.join(test_dir, fname))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (img_size, img_size))
        images.append(img)
    return np.stack(images), image_files

def yolo_inference(model_path: str, images: np.ndarray, conf: float, iou: float, max_det: int) -> list:
    """
    Run inference using the YOLO model.
    Args:
        model_path (str): Path to YOLO weights.
        images (np.ndarray): Batch of images (N, H, W, C).
        conf (float): Confidence threshold.
        iou (float): IoU threshold.
        max_det (int): Maximum detections per image.
    Returns:
        list: List of YOLO results.
    """
    model = YOLO(model_path)
    results = model.predict(
        images,
        conf=conf,
        iou=iou,
        max_det=max_det,
        verbose=False
    )
    return results

def onnx_inference(onnx_path: str, images: np.ndarray) -> list:
    """
    Run inference using the exported ONNX model.
    Args:
        onnx_path (str): Path to ONNX model.
        images (np.ndarray): Batch of images (N, H, W, C).
    Returns:
        list: List of ONNX outputs.
    """
    session = ort.InferenceSession(onnx_path)
    input_name = session.get_inputs()[0].name
    # Normalize and transpose images to NCHW, float32
    images_nchw = images.transpose(0, 3, 1, 2).astype(np.float32) / 255.0
    outputs = session.run(None, {input_name: images_nchw})
    return outputs

# Load test images
images, image_files = load_test_images(config.TEST_IMAGES_DIR, config.IMG_SIZE)

# YOLO inference
print("Running YOLO inference on test set...")
yolo_results = yolo_inference(
    config.MODEL_PATH, images, config.CONF_THRESHOLD, config.IOU_THRESHOLD, config.MAX_DET
)

# ONNX inference
print("Running ONNX inference on test set...")
onnx_outputs = onnx_inference(config.ONNX_EXPORT_PATH, images)


In [ ]:
# 4. Compare YOLO and ONNX results on the test set

def compare_yolo_onnx(yolo_results: list, onnx_outputs: list, image_files: list) -> None:
    """
    Compare YOLO and ONNX outputs for consistency.
    Args:
        yolo_results (list): YOLO model results.
        onnx_outputs (list): ONNX model outputs.
        image_files (list): List of image filenames.
    """
    print("\nComparison of YOLO vs ONNX outputs (first 5 images):")
    for idx in range(min(5, len(image_files))):
        print(f"Image: {image_files[idx]}")
        # YOLO: number of detections
        yolo_num_det = len(yolo_results[idx].boxes) if hasattr(yolo_results[idx], 'boxes') else 0
        print(f"  YOLO detections: {yolo_num_det}")
        # ONNX: shape of output (batch, num_boxes, ...)
        onnx_num_det = onnx_outputs[0][idx].shape[0] if len(onnx_outputs[0].shape) == 3 else 0
        print(f"  ONNX output boxes: {onnx_num_det}")
        print("-")

compare_yolo_onnx(yolo_results, onnx_outputs, image_files)